# M4: CLV 조건부 다중 음성 BPR (Dunnhumby, seed 42)

M1의 임베딩·이진 그래프·LightGCN 전파는 그대로 두고 손실만 바꿉니다. 양성상품마다 동일한 uniform 음성 5개를 사용하여 `K=5 평균 BPR` 대조군과 `historical CLV가 높을수록 최고점 음성에 집중하는 BPR`을 비교합니다. day 684~690 개발분할만 평가하며 최종 test와 holdout은 생성하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, subprocess

REVIEWED_SHA = '6c3ca0b2ce3199374fada636f4954ee13add81c4'
repo = Path('/content/clv-m2-lightgcn-runner')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)

In [ ]:
import json
import torch
from lightgcn_clv_m4_clv_hard_negative import (
    configure_m4_clv_hard_negative_run,
    preflight_summary,
    run_m4_clv_hard_negative_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_m4_clv_hard_negative_run()
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))

In [ ]:
result_df = run_m4_clv_hard_negative_screen(cfg)

In [ ]:
from IPython.display import display
import pandas as pd

print('1) 절대지표')
display(result_df)
print('2) M1 및 K=5 평균 대조군 대비 비교')
display(pd.DataFrame(result_df.attrs['comparison']))
print('3) 사전 판정 규칙 결과')
print(json.dumps(result_df.attrs['screening_reading'], ensure_ascii=False, indent=2))
print('4) 학습 작동 진단')
print(json.dumps(result_df.attrs['training_diagnostics'], ensure_ascii=False, indent=2))
print('5) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))